# 🔍 RAG — End-to-End Retrieval Pipeline
### AI/ML Bootcamp — Day 2

```
Document  →  LangChain Loader  →  RecursiveCharacterTextSplitter  →  Embeddings  →  ChromaDB
                                                                                        ↓
User Query  ──────────────────────────────────────────────────────  Retrieve Top-K Chunks
                                                                                        ↓
                                                               Build Prompt  →  LLM  →  Answer
```

**Stack:**
- `langchain` — document loader + recursive text splitter
- `sentence-transformers` — local embeddings (all-MiniLM-L6-v2)
- `chromadb` — vector store
- `openai` — LLM (gpt-4o-mini)

---

## Step 1 — Install & Imports

In [ ]:
!pip install openai chromadb sentence-transformers langchain langchain-community -q

import os
import shutil
import chromadb
from openai import OpenAI
from sentence_transformers import SentenceTransformer
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ── API Key ───────────────────────────────────────
OPENAI_API_KEY = "sk-..."   # paste your key here
# ─────────────────────────────────────────────────

openai_client = OpenAI(api_key=OPENAI_API_KEY)

print("✅ Ready")

---
## Step 2 — Write & Load the Document

We write a `.txt` file to disk and load it using LangChain's `TextLoader`.  
You can replace this with any file path you already have.

In [ ]:
# Write the document to disk
DOC_PATH = "novatech.txt"

with open(DOC_PATH, "w") as f:
    f.write("""NovaTech AI — Company Knowledge Base

About the Company
NovaTech AI is a Karachi-based artificial intelligence startup founded in 2021 by Fatima Al-Rashid (CEO)
and Bilal Mahmood (CTO). The company specializes in enterprise AI solutions for logistics and supply chain.
As of 2024, NovaTech employs over 120 engineers across offices in Karachi, Dubai, and London.
The company raised a $47 million Series B funding round in March 2024, led by Sequoia Capital.

Products
LogiSense is NovaTech's flagship AI-powered logistics optimization platform. It uses real-time data from
IoT sensors, weather APIs, and traffic systems to recommend optimal delivery routes. Clients report an
average 23% reduction in delivery costs. LogiSense processes over 2 million delivery events per day.

DocuMind is an intelligent document processing tool that uses large language models to extract, classify,
and summarize information from invoices, contracts, and shipping documents. It supports 14 languages and
achieves 97.3% extraction accuracy. DocuMind integrates with SAP, Oracle, and Microsoft Dynamics.

SupplyIQ is a supply chain intelligence dashboard providing predictive analytics for inventory management,
demand forecasting, and supplier risk assessment. It uses gradient boosting models trained on 5 years of
historical data. SupplyIQ requires at least 6 months of historical data to generate baseline forecasts.

Pricing and Plans
Standard plan: $499/month. Includes access to LogiSense and DocuMind, up to 50,000 API calls/month,
and business-hours email support with a 99.5% uptime SLA.

Professional plan: $1,499/month. Includes all Standard features plus SupplyIQ, 200,000 API calls/month,
priority support, and a dedicated customer success manager.

Enterprise plan: Custom pricing. Includes unlimited API calls, 99.9% uptime SLA, 24/7 dedicated support,
on-premise deployment option, and custom model fine-tuning.

Support and SLA
Enterprise customers receive a 99.9% uptime SLA with 24/7 dedicated support. Critical incidents are
resolved within 4 hours for enterprise tier. Standard tier customers receive a 99.5% uptime SLA with
business-hours support and 24-hour incident resolution. All customers can access the developer portal,
API documentation, and community forum at docs.novatech.io.

Security
NovaTech is ISO 27001 certified and GDPR compliant. All customer data is encrypted at rest using
AES-256 and in transit using TLS 1.3. The company undergoes quarterly penetration testing. PII data
is anonymized at ingestion before entering any ML training pipeline. Data is retained for 12 months
by default. Enterprise customers can configure custom retention from 30 days to 7 years.

Technology Stack
NovaTech's ML infrastructure is built on PyTorch and deployed on AWS using Kubernetes. Models are
versioned with MLflow and served through FastAPI endpoints. Real-time pipelines use Apache Kafka.
Batch pipelines run on Apache Airflow. The data warehouse is Snowflake.
""")

# ── LangChain TextLoader ──────────────────────────────────────
loader = TextLoader(DOC_PATH, encoding="utf-8")
docs   = loader.load()
# ─────────────────────────────────────────────────────────────

print(f"Pages loaded  : {len(docs)}")
print(f"Characters    : {len(docs[0].page_content):,}")
print(f"Metadata      : {docs[0].metadata}")
print(f"\nPreview:\n{docs[0].page_content[:300]}")

---
## Step 3 — Chunking with RecursiveCharacterTextSplitter

`RecursiveCharacterTextSplitter` tries to split on `\n\n` first (paragraphs),  
then `\n` (lines), then `.` (sentences), then spaces — in that order.  
It stops as soon as chunks are within `chunk_size`.

In [ ]:
# ── LangChain RecursiveCharacterTextSplitter ──────────────────
splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,        # max characters per chunk
    chunk_overlap=50,      # overlap between consecutive chunks
    separators=["\n\n", "\n", ". ", " ", ""],  # priority order
)

chunks = splitter.split_documents(docs)
# ─────────────────────────────────────────────────────────────

print(f"Total chunks: {len(chunks)}")
print()

for i, chunk in enumerate(chunks):
    print(f"--- Chunk {i+1} ({len(chunk.page_content)} chars) ---")
    print(chunk.page_content)
    print()

---
## Step 4 — Embeddings

Encode every chunk into a 384-dimensional vector using `all-MiniLM-L6-v2`.

In [ ]:
print("Loading embedding model (downloads ~90MB on first run)...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")

texts            = [chunk.page_content for chunk in chunks]
chunk_embeddings = embedder.encode(texts, normalize_embeddings=True, show_progress_bar=True)

print(f"\nEmbeddings: {chunk_embeddings.shape[0]} chunks × {chunk_embeddings.shape[1]} dims")

---
## Step 5 — Store in ChromaDB

In [ ]:
# Fresh collection every run
if os.path.exists("./chroma_db"):
    shutil.rmtree("./chroma_db")

chroma_client = chromadb.PersistentClient(path="./chroma_db")
collection    = chroma_client.create_collection(
    name="novatech",
    metadata={"hnsw:space": "cosine"}
)

collection.add(
    ids        = [f"chunk_{i}" for i in range(len(texts))],
    embeddings = chunk_embeddings.tolist(),
    documents  = texts,
    metadatas  = [chunk.metadata for chunk in chunks],   # source file, page, etc.
)

print(f"✅ Stored {collection.count()} chunks in ChromaDB")

---
## Step 6 — Retriever

Embed the user query with the **same model** and find the closest chunks.

In [ ]:
def retrieve(query, top_k=3):
    query_embedding = embedder.encode([query], normalize_embeddings=True).tolist()

    results = collection.query(
        query_embeddings=query_embedding,
        n_results=top_k
    )

    retrieved = []
    for doc, distance in zip(results["documents"][0], results["distances"][0]):
        retrieved.append({
            "text" : doc,
            "score": round(1 - distance, 4)   # distance → similarity
        })

    return retrieved


# Quick test
results = retrieve("What is the enterprise plan?", top_k=3)

print("Query: What is the enterprise plan?\n")
for i, r in enumerate(results, 1):
    print(f"[{i}] score={r['score']}")
    print(f"    {r['text']}")
    print()

---
## Step 7 — Build the Prompt

Concatenate retrieved chunks as context, then append the user question.

In [ ]:
def build_prompt(query, retrieved_chunks):
    context = "\n\n".join(
        f"[Chunk {i+1}]\n{r['text']}"
        for i, r in enumerate(retrieved_chunks)
    )

    return f"""You are a helpful assistant. Answer the question using ONLY the context below.
If the answer is not in the context, say "I don't have that information."

Context:
{context}

Question: {query}

Answer:"""


# Preview
sample_prompt = build_prompt(
    "What is the enterprise plan?",
    retrieve("What is the enterprise plan?", top_k=3)
)
print(sample_prompt)

---
## Step 8 — LLM Call

Send the prompt to `gpt-4o-mini` and return the answer.

In [ ]:
def ask(query, top_k=3, verbose=False):
    # 1. Retrieve relevant chunks
    retrieved = retrieve(query, top_k=top_k)

    # 2. Build prompt
    prompt = build_prompt(query, retrieved)

    # 3. Call the LLM
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=300,
    )

    answer = response.choices[0].message.content.strip()

    if verbose:
        print("Retrieved chunks:")
        for i, r in enumerate(retrieved, 1):
            print(f"  [{i}] score={r['score']}  {r['text'][:100]}...")
        print(f"Tokens used: {response.usage.total_tokens}\n")

    return answer


answer = ask("What is the enterprise plan?", verbose=True)
print("Answer:", answer)

---
## Step 9 — Test with Multiple Questions

In [ ]:
questions = [
    "Who founded NovaTech and when?",
    "What products does NovaTech offer?",
    "What is the uptime SLA for standard customers?",
    "How does DocuMind handle document extraction?",
    "Is NovaTech GDPR compliant?",
    "What technology does NovaTech use for ML?",
    "What is the price of the Professional plan?",
    "What is the capital of France?",   # out-of-context — should say 'I don't have that information'
]

for q in questions:
    print(f"Q: {q}")
    print(f"A: {ask(q)}")
    print()

---
## Step 10 — Interactive Chat Loop

In [ ]:
print("NovaTech AI Assistant  (type 'quit' to stop)")
print("-" * 45)

while True:
    query = input("You: ").strip()
    if query.lower() in ("quit", "exit", "q"):
        break
    if query:
        print(f"Bot: {ask(query)}\n")

---
## What each piece does

| Step | Tool | What it does |
|---|---|---|
| Load | `LangChain TextLoader` | Reads the file → `Document(page_content, metadata)` |
| Chunk | `RecursiveCharacterTextSplitter` | Splits on `\n\n` → `\n` → `.` → space, respects boundaries |
| Embed | `SentenceTransformer` | Converts text → 384-dim vector |
| Store | `ChromaDB` | Indexes vectors for fast cosine similarity search |
| Retrieve | `collection.query()` | Embeds query → finds top-K closest chunks |
| Prompt | `build_prompt()` | Wraps chunks + question into a single string |
| Answer | `gpt-4o-mini` | Reads context, answers only from what's there |